In [ ]:
import numpy as np
import pandas as pd

from imblearn.combine import SMOTEENN
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from models.MLPipeline import *
from utils import ASSETS_DIR

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import KNNImputer

import xgboost as xgb
from xgboost import XGBClassifier


In [ ]:
df = pd.read_parquet(ASSETS_DIR / 'final_df.parquet')

In [ ]:



X = df.drop(columns=['TARGET'])
X.drop(columns=['LBDEVAL'], inplace=True, errors='ignore')
y = df['TARGET']



X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Buchi (NaN) iniziali in X_train: {X_train.isna().sum().sum()}")



imputer = KNNImputer(n_neighbors=7, weights='distance')


X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)


X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)


X_train_imp = X_train_imp.round()
X_test_imp = X_test_imp.round()

print(f"Buchi (NaN) finali in X_train: {X_train_imp.isna().sum().sum()}")
print(f"Buchi (NaN) finali in X_test: {X_test_imp.isna().sum().sum()}")

In [ ]:
scaler = MinMaxScaler()

x_train_scaled = scaler.fit_transform(X_train_imp)
x_test_scaled = scaler.transform(X_test_imp)

In [ ]:
xgb_base = XGBClassifier(
    objective='multi:softprob', # Ottimizzato per il multiclasse (restituisce probabilità)
    num_class=3,
    tree_method='hist',         # Accelera drasticamente la Grid Search
    random_state=42,
    n_jobs=-1,                   # Usa tutti i core del processore
    booster="gtree"
)
smote_enn = SMOTEENN(random_state=42)
smote_nc = BorderlineSMOTE(random_state=42)
pipeline_xgb = ImbPipeline(steps=[
    ('smoteenn', smote_nc),
    ('Classifier', model)
])

y_true_bin, y_proba = train_model_evaluate(x_train_scaled, y_train, pipeline_xgb)

In [ ]:
generate_predictions_and_cm(x_train_scaled, y_train, pipeline_xgb)
plot_reliability_diagram(y_train, y_proba[:, 2], title='Reliability Diagram - Random Forest (Train Set)')

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTEENN
import time
from sklearn.model_selection import ParameterGrid

print("Configurazione dell'architettura XGBoost + GridSearch...")

# 1. Inizializziamo SMOTEENN e il modello base XGBoost
smote_enn = SMOTEENN(random_state=42)

# Usiamo tree_method='hist' che è brutalmente più veloce per dataset grandi

xgb_base = XGBClassifier(
    objective='multi:softprob', # Ottimizzato per il multiclasse (restituisce probabilità)
    num_class=3,
    tree_method='hist',         # Accelera drasticamente la Grid Search
    random_state=42,
    n_jobs=-1,                   # Usa tutti i core del processore
    booster="gtree"
)

# 2. Creiamo la Pipeline Ibrida (Il vero segreto di questa architettura)
pipeline_xgb = ImbPipeline(steps=[
    ('smoteenn', smote_enn),
    ('xgb', xgb_base)
])

# 3. Definiamo la Griglia degli Iperparametri
# ATTENZIONE: Essendo dentro una pipeline, dobbiamo usare il prefisso 'xgb__' 
# (il nome che abbiamo dato allo step) per puntare ai parametri del classificatore.
param_grid = {
    'xgb__n_estimators': [100, 200],         # Numero di alberi
    'xgb__max_depth': [3, 5, 7],             # Profondità (3=sicuro, 7=rischio overfitting)
    'xgb__learning_rate': [0.01, 0.05],       # Quanto ogni albero corregge il precedente (eta)
    'xgb__subsample': [0.8, 1.0],            # % di pazienti usati per ogni albero (aiuta la generalizzazione)
    'xgb__colsample_bytree': [0.8, 1.0],      # % di feature usate per ogni albero
    
}

# 4. Inizializziamo GridSearchCV con la tua StratifiedKFold (cv_strategy)
grid_search = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=param_grid,
    cv=cv_strategy,           # La StratifiedKFold(n_splits=5) definita in precedenza
    scoring='f1_macro',       # Usiamo F1 Macro per dare peso al Lewy Body (Classe 2)
    n_jobs=-1,
    verbose=2,                # Mostra l'avanzamento (utile perché ci vorrà un po')
    return_train_score=False
)

print(f"Avvio Grid Search: {len(ParameterGrid(param_grid))} combinazioni x 5 Folds...")
start_time = time.time()

# 5. Il Fit Definitivo (usa i tuoi dati scalati e traslati puliti)
# Sostituisci X_train_translated con il nome effettivo del tuo dataset preprocessato
grid_search.fit(x_train_scaled, y_train)

end_time = time.time()
print(f"\nRicerca completata in {(end_time - start_time)/60:.2f} minuti.")

# 6. Estrazione dei risultati vincenti
print("\n=== RISULTATI GRID SEARCH XGBOOST ===")
print(f"Miglior F1-Score (Macro) in Cross-Validation: {grid_search.best_score_:.4f}")
print("Migliori Iperparametri trovati:")
for param, value in grid_search.best_params_.items():
    print(f" - {param.replace('xgb__', '')}: {value}")

# Salviamo il modello vincente (la pipeline completa addestrata con i parametri ottimali)
best_xgb_pipeline = grid_search.best_estimator_


In [ ]:
y_true_bin, y_proba = train_model_evaluate(X=X_train_imp, model=best_xgb_pipeline, y=y_train)

generate_predictions_and_cm(X_train_imp, y_train, best_xgb_pipeline)
plot_reliability_diagram(y_true_bin, y_proba[:, 2], title='Reliability Diagram - XGBoost')
